In [ ]:
import os
import pandas as pd
from BERTopic_model import run_BERTopic_model

# Dataloading
df = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "data", "data_advice_fulltext.csv"))
docs = list(df["text"])

# hyperparameters setting
base_param_dic = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "n_neighbors": 15,
    "n_components": 10,
    "min_dist": 0.0,
    "min_cluster_size": 20,
    "min_df": 3,
    "max_df": 1.0,
    "ngram_range": (1, 3),
    "top_n_words": 5,
    "seed": 37
}

all_param_dic = {
    "n_neighbors": [5, 30],
    "n_components": [5, 15],
    "min_dist": [0.1, 0.5],
    "min_cluster_size": [10, 30],
    "min_df": [0.0, 10],
    "max_df": [0.5, 0.8],
    "ngram_range": [(1, 1), (1, 5)],
    "top_n_words": [10, 20]
}

# run



for key, values in all_param_dic.items():
    for value in values:
        run_name = f'{"bertopic"}_{key}-{str(value)}'
        param_dic = base_param_dic.copy()
        param_dic[key] = value
        topic_model = run_BERTopic_model(param_dic, docs)
        topic_model.save(os.path.join(os.path.dirname(os.getcwd()), "results", "robustness_check", run_name), serialization="safetensors", save_ctfidf=True, save_embedding_model="all-MiniLM-L6-v2")

## Loading

In [1]:
from bertopic import BERTopic
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer

import os
import pandas as pd
from BERTopic_model import run_BERTopic_model

# Dataloading
df = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "data", "data_advice_fulltext.csv"))
docs = list(df["text"])

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", use_auth_token=False)
embeddings = embedding_model.encode(docs, show_progress_bar=True)



Batches:   0%|          | 0/32 [00:00<?, ?it/s]

## Similarity (not working)

In [43]:
base_model = BERTopic.load(os.path.join(os.path.dirname(os.getcwd()), "results", "BERTopic_model"), embedding_model=embedding_model)
base_topic_docs = [doc for doc, topic in zip(docs, base_model.topics_) if topic == 1]
base_embeddings = base_model._extract_embeddings(base_topic_docs)


curr_model = BERTopic.load(os.path.join(os.path.dirname(os.getcwd()), "results", "robustness_check", "bertopic_min_cluster_size-30"), embedding_model=embedding_model)
curr_topic_docs = [doc for doc, topic in zip(docs, curr_model.topics_) if topic == 2]
curr_embeddings = curr_model._extract_embeddings(curr_topic_docs)

np.mean(cosine_similarity(base_embeddings, curr_embeddings))

KeyboardInterrupt: 

In [ ]:
curr_model.get_topic_info()

In [ ]:


curr_topic_docs = [doc for doc, topic in zip(docs, topic_model.topics_) if topic == 1]
curr_embeddings = topic_model._extract_embeddings(curr_topic_docs)
similarity_matrix = cosine_similarity(base_embeddings, curr_embeddings)
similarity = np.mean(similarity_matrix)
similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

topic_distr, _ = topic_model.approximate_distribution(docs)


topic_docs = [doc for doc, topic in zip(docs, topic_model.topics_) if topic == 1]
embeddings = topic_model._extract_embeddings(topic_docs)
similarity_matrix = cosine_similarity(embeddings)
average_similarity = np.mean(similarity_matrix)
average_similarity

## Correlation

In [31]:
def compute_topic_corr(topic_model, docs, var):

    topic_distr, _ = topic_model.approximate_distribution(docs)
    df_stat = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "data", "data_stake.csv"))

    for topic_n in range(len(topic_distr[0,:])):
        topic_name = "topic_" + str(topic_n)
        df_stat[topic_name] = topic_distr[:, topic_n]

    df_stat["assigned_topic"] = topic_model.topics_
    return df_stat["pl2_understanding"].corr(df_stat[var])

In [50]:
run_name = []
corr = []
n = []

base_model = BERTopic.load(os.path.join(os.path.dirname(os.getcwd()), "results", "BERTopic_model"), embedding_model=embedding_model)

corr.append(compute_topic_corr(base_model, docs, "topic_1"))
run_name.append("base")
n.append(base_model.topics_.count(1))

results_dir = os.path.join(os.path.dirname(os.getcwd()), "results", "robustness_check")
robustness_models = [f for f in os.listdir(results_dir) if os.path.isdir(os.path.join(results_dir, f))]

# arbitrary topic number, depends on the specific model, decision made by human inspection
topic_n = [1, 1, 0, 2, 1, 1, 2, 2, 1, 1, 1, 1, 2, 1, 1, 1]

i = 0
for model_name in robustness_models:
    topic_model = BERTopic.load(os.path.join(os.path.dirname(os.getcwd()), "results", "robustness_check", model_name), embedding_model=embedding_model)
    corr.append(compute_topic_corr(topic_model, docs, "topic_" + str(topic_n[i])))
    run_name.append(model_name)
    n.append(topic_model.topics_.count(topic_n[i]))
    i += 1


100%|██████████| 1/1 [00:00<00:00,  3.26it/s]


108

## Data saving

In [55]:
data = pd.DataFrame({
    "run_name": run_name,
    "corr": corr,
    "n": n
})

data.to_csv(os.path.join(os.path.dirname(os.getcwd()), "results", "robustness_check", "results.csv"), header=True, index=False)

## Dump

In [ ]:
df = pd.DataFrame({
    "run_name": [run_name],
    "corr": [0.83],
    "similarity": [0.7],
    "size": sum(df_stat["assigned_topic"] == 1)
    })

df